In [1]:
import pandas as pd
from catboost import CatBoostRegressor, Pool, cv
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import numpy as np

In [2]:
cols = ['bedrooms', 'beds', 'baths', 'luxury_items', 'luxury_score', 'location_desirability']

In [75]:
df = pd.read_parquet('./dataset/forCatboost.parquet')
df = df.loc[:, ['Price'] + cols +['quality', 'city']].astype(np.float64, errors='ignore')
vc = df['city'].value_counts()
vc =  vc[vc >= 14]
cities = vc.index
df = df[df['city'].isin(cities)].reset_index(drop=True)
df['quality'] = np.where(df['quality'] == 'acceptable', 'great', df['quality'])
# df = df[df['quality']!='acceptable']
df.info() 

<class 'pandas.DataFrame'>
RangeIndex: 1474 entries, 0 to 1473
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Price                  1474 non-null   float64
 1   bedrooms               1474 non-null   float64
 2   beds                   1474 non-null   float64
 3   baths                  1474 non-null   float64
 4   luxury_items           1474 non-null   float64
 5   luxury_score           1474 non-null   float64
 6   location_desirability  1474 non-null   float64
 7   quality                1474 non-null   str    
 8   city                   1474 non-null   str    
dtypes: float64(7), str(2)
memory usage: 123.4 KB


In [38]:
df['quality'].value_counts()

quality
great      1409
perfect      60
Name: count, dtype: int64

In [76]:
X = df.iloc[:, 1:]
y = df['Price']

In [82]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [77]:
p = Pool(X, y, cat_features=['city', 'quality'])
params = {
  'cat_features': ['city', 'quality'],
  'depth': 6,
  'loss_function': 'RMSE',
  'iterations': 400
  
}

cv_res = cv(p, params, fold_count=5)
cv_res

Training on fold [0/5]
0:	learn: 125941.8192964	test: 124199.5767531	best: 124199.5767531 (0)	total: 27.6ms	remaining: 11s
1:	learn: 123344.3749453	test: 121785.3791199	best: 121785.3791199 (1)	total: 49.9ms	remaining: 9.94s
2:	learn: 120749.9327364	test: 119241.0696981	best: 119241.0696981 (2)	total: 81.8ms	remaining: 10.8s
3:	learn: 118258.9856440	test: 117077.3712157	best: 117077.3712157 (3)	total: 107ms	remaining: 10.6s
4:	learn: 115951.7042480	test: 115104.7399123	best: 115104.7399123 (4)	total: 139ms	remaining: 11s
5:	learn: 113641.1254541	test: 112883.1020470	best: 112883.1020470 (5)	total: 168ms	remaining: 11s
6:	learn: 111330.8136797	test: 110694.0019401	best: 110694.0019401 (6)	total: 208ms	remaining: 11.7s
7:	learn: 109144.8533345	test: 108554.9967881	best: 108554.9967881 (7)	total: 248ms	remaining: 12.1s
8:	learn: 107071.5578947	test: 106599.6156628	best: 106599.6156628 (8)	total: 285ms	remaining: 12.4s
9:	learn: 105009.5910589	test: 104755.9307158	best: 104755.9307158 (9)	

,iterations,test-RMSE-mean,test-RMSE-std,train-RMSE-mean,train-RMSE-std
0,0,125427.755224,6738.411351,125511.834777,1726.741488
1,1,122926.883710,6856.889795,122961.669705,1696.412620
2,2,120374.451594,6904.128786,120379.525944,1700.398513
3,3,118029.558611,6984.814972,117964.223940,1691.546585
4,4,115737.900413,7053.172041,115627.561300,1682.477736
...,...,...,...,...,...
395,395,47109.735535,7605.893058,34399.960967,1650.945437
396,396,47111.061654,7610.468306,34376.134705,1641.283626
397,397,47087.516245,7614.183002,34336.471241,1636.224186
398,398,47078.290236,7619.705273,34296.644765,1621.704026


In [86]:
model_final = CatBoostRegressor(cat_features=['city', 'quality'], depth=6)
model_final.fit(X, y)

Learning rate set to 0.043531
0:	learn: 76413.6064421	total: 27.2ms	remaining: 27.2s
1:	learn: 75119.6936730	total: 56.3ms	remaining: 28.1s
2:	learn: 73895.6562521	total: 84.7ms	remaining: 28.2s
3:	learn: 72699.1110674	total: 113ms	remaining: 28.2s
4:	learn: 71404.0436416	total: 140ms	remaining: 27.8s
5:	learn: 70293.6186450	total: 167ms	remaining: 27.6s
6:	learn: 69090.7260326	total: 196ms	remaining: 27.8s
7:	learn: 68163.7723366	total: 226ms	remaining: 28s
8:	learn: 67172.6917530	total: 256ms	remaining: 28.2s
9:	learn: 66483.9059543	total: 271ms	remaining: 26.8s
10:	learn: 65802.0839843	total: 301ms	remaining: 27.1s
11:	learn: 65000.4351590	total: 331ms	remaining: 27.3s
12:	learn: 64425.9900676	total: 361ms	remaining: 27.4s
13:	learn: 63685.1341482	total: 389ms	remaining: 27.4s
14:	learn: 62888.0365788	total: 419ms	remaining: 27.5s
15:	learn: 62146.2130793	total: 449ms	remaining: 27.6s
16:	learn: 61346.1901891	total: 479ms	remaining: 27.7s
17:	learn: 60491.4190528	total: 507ms	remain

CatBoostRegressor(cat_features=['city', 'quality'], depth=6, loss_function='RMSE')

<h1>Optimistic Evaluation</h1>

In [87]:
ypred = model_final.predict(X)

print(f'MAE: {mean_absolute_error(y, ypred)}')
print(f'MAPE: {mean_absolute_percentage_error(y, ypred)}')

MAE: 18795.05727351649
MAPE: 0.287915681830392


In [88]:
model_final.save_model('./GUI/airbnb_catboost.cmb')

In [89]:
X_test.to_csv('test.csv', index=False)